# Deliverable 2 — Common state-encoding audit

Tests raw-population and square-root-population encodings against the same Carleman collision and streaming operations.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Question

The legacy streaming notebook uses amplitudes proportional to \(\sqrt f\),
whereas the Carleman operator is linear in a lifted vector made from \(f\).
This experiment determines which representation is algebraically compatible
with collision and verifies that streaming remains a permutation in it.

In [2]:
import json
import numpy as np
import pandas as pd
from quantum_aero.classical import LBMConfig, stream
from quantum_aero.carleman import lift, lifted_matrix, order2_collision
from quantum_aero.deliverables import initial_lattice_state

cfg = LBMConfig(n=4, reynolds=100, t_end=0.1, mach=0.05, snapshots=2)
field, omega, velocity_scale, dt = initial_lattice_state(cfg)
f = field[1, 2].copy()
matrix = lifted_matrix(omega)
expected = order2_collision(f, omega)

raw_result = (matrix @ lift(f))[:9]
sqrt_f = np.sqrt(np.clip(f, 0, None))
sqrt_amplitude_result = (matrix @ lift(sqrt_f))[:9]
sqrt_decoded = np.square(sqrt_amplitude_result)

rows = [
    {"encoding": "raw f amplitudes", "collision_relative_error": np.linalg.norm(raw_result-expected)/np.linalg.norm(expected)},
    {"encoding": "sqrt(f) amplitudes then square", "collision_relative_error": np.linalg.norm(sqrt_decoded-expected)/np.linalg.norm(expected)},
]
pd.DataFrame(rows)

,encoding,collision_relative_error
0,raw f amplitudes,1.433910e-16
1,sqrt(f) amplitudes then square,4.914641e+00


In [3]:
raw_stream = stream(field).reshape(-1)
sqrt_stream_decoded = np.square(np.sqrt(np.clip(field, 0, None))[..., :])
sqrt_stream_decoded = stream(sqrt_stream_decoded).reshape(-1)
stream_error = np.max(np.abs(raw_stream - sqrt_stream_decoded))

result = {
    "chosen_encoding": "raw population amplitudes with an explicit norm/scale register or oracle",
    "raw_collision_relative_error": rows[0]["collision_relative_error"],
    "sqrt_collision_relative_error": rows[1]["collision_relative_error"],
    "streaming_representation_error": float(stream_error),
    "conclusion": "streaming supports either representation, but the Carleman collision requires raw-f lifted amplitudes",
}
(output_dir / "02_common_state_encoding.json").write_text(json.dumps(result, indent=2))
assert result["raw_collision_relative_error"] < 1e-12
assert result["sqrt_collision_relative_error"] > 1e-3
print(json.dumps(result, indent=2))

{
  "chosen_encoding": "raw population amplitudes with an explicit norm/scale register or oracle",
  "raw_collision_relative_error": 1.4339096366825125e-16,
  "sqrt_collision_relative_error": 4.914640544334296,
  "streaming_representation_error": 1.3877787807814457e-17,
  "conclusion": "streaming supports either representation, but the Carleman collision requires raw-f lifted amplitudes"
}


## Decision

Use amplitudes proportional to signed/raw populations for the proposed
Carleman path. The remaining circuit-level requirement is a reversible
preparation oracle that also exposes the normalization needed for engineering
observables; probability decoding from the legacy \(\sqrt f\) notebook cannot
be reused unchanged.